In [6]:
cd /home/nampv1/projects/asr/asr_ft

/home/nampv1/projects/asr/asr_ft


In [7]:
from src.utils.model_utils import load_whisper_model, load_processor

/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
import os
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, AutoProcessor

device_args = {}
device_args['use_cpu'] = False

def merge_adapter(adapter_dir, merged_dir, model_name_or_path):
    """
    Merges a PEFT adapter into a base Whisper model and saves the merged model and processor.

    Parameters:
    - adapter_dir (str): Path to the directory containing the adapter.
    - merged_dir (str): Path to save the merged model and processor.
    - model_name_or_path (str): Path or identifier of the base Whisper model.
    - device (str, optional): Device to load the model on ('cpu' or 'cuda'). Default is 'cpu'.
    """
    # Load base model and processor
    base_model = WhisperForConditionalGeneration.from_pretrained(model_name_or_path)
    processor = AutoProcessor.from_pretrained(model_name_or_path)

    # Load PEFT adapter and merge
    finetuned_model = PeftModel.from_pretrained(base_model, adapter_dir)
    # finetuned_model = finetuned_model.merge_and_unload()

    # Save merged model and processor
    os.makedirs(merged_dir, exist_ok=True)
    finetuned_model.save_pretrained(merged_dir)
    processor.save_pretrained(merged_dir)

    print(f"Merged model and processor saved to {merged_dir}")
    return merged_dir


In [18]:
merge_adapter(
    adapter_dir='draft/checkpoint-4000',
    merged_dir='draft/merged_model',
    model_name_or_path='openai/whisper-large-v3-turbo',
)


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 15141.89it/s]


Merged model and processor saved to draft/merged_model


'draft/merged_model'

In [12]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [13]:
from transformers import WhisperForConditionalGeneration
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3-turbo")

In [20]:
merged_model = WhisperForConditionalGeneration.from_pretrained("draft/merged_model")

In [14]:
print_trainable_parameters(base_model)

trainable params: 806958080 || all params: 808878080 || trainable%: 99.76263419080414


In [21]:
print_trainable_parameters(merged_model)

trainable params: 0 || all params: 815431680 || trainable%: 0.0


In [28]:
(815431680 - 808878080) / 815431680, 815431680 - 808878080

(0.008036970062286517, 6553600)

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel
import os

model_name = "openai/whisper-large-v3-turbo"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Nếu có adapter_path trong config thì merge adapter LoRA
adapter_paths = "/home/nampv1/projects/asr/asr-deployment-app/models/adapters/on_data1/checkpoint-7434,/home/nampv1/projects/asr/asr-deployment-app/models/adapters/on_data2/checkpoint-6500"
adapter_paths = [p.strip() for p in adapter_paths.split(",") if p.strip()]

if adapter_paths:
    if isinstance(adapter_paths, str):
        adapter_paths = [adapter_paths]  # convert to list if single path

    for path in adapter_paths:
        model = PeftModel.from_pretrained(model, path)
        model = model.merge_and_unload()

merged_dir = "/home/nampv1/projects/asr/asr-deployment-app/models/merged/vnpost_asr_01_20250920"

os.makedirs(merged_dir, exist_ok=True)
model.save_pretrained(merged_dir)
processor.save_pretrained(merged_dir)